# 01 · Workflow — run any single node

<div style="background: rgba(128, 128, 128, 0.1); border-left: 4px solid rgba(128, 128, 128, 0.6); padding: 8px 14px; border-radius: 6px;">

This notebook is the day-to-day loop for **one node**, and the main way to run nodes in IQCC Learn:

1. download the **latest QUAM state and wiring** from IQCC,
2. pick **any node** from the library (default `05_T1`) — it runs on a random active qubit,
3. look at the plot, the fit and the **state updates the node proposes** — the data is always saved, but the state is **not** changed yet,
4. optionally **accept** the proposed updates (all or some), or leave the state as it is.

To chain several nodes, a graph notebook will follow. If you have not yet seen what a node is made of, start with `00_start_here.ipynb`.

</div>

<div style="background: rgba(46, 160, 67, 0.1); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 Packages: **QUAlibrate** ([docs](https://qua-platform.github.io/qualibrate/)) runs the nodes; **QUAM** ([docs](https://qua-platform.github.io/quam/)) holds the state they read and update;
💎 **QUA** ([docs](https://docs.quantum-machines.co/latest/)) is the program each node runs on the OPX.

</div>

## 1 · Pick a node

<div style="background: rgba(46, 160, 67, 0.1); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAlibrate** — [docs](https://qua-platform.github.io/qualibrate/) · `QualibrationLibrary` scans `calibrations/` and registers every node by name.

</div>

Run the next cell to list the available nodes, then copy the one you want into `NODE` in step 2.

In [ ]:
from qualibrate import QualibrationLibrary

library = QualibrationLibrary.get_active_library()
print("available nodes:")
for name in sorted(library.nodes):
    print("  ", name)

## 2 · Settings

In [ ]:
NODE = "05_T1"      # copy any name from the list above
QUBITS = None       # None: one random active qubit of the machine; or e.g. ["q1", "q2"]
SIMULATE = False    # True: run on the OPX simulator instead of the hardware

## 3 · Backend and state location
Read from the active qualibrate config, the same source the nodes use, so it is the same in every checkout.

In [ ]:
from pathlib import Path

# Everything below comes from the active qualibrate config: the same source the nodes use at runtime.
from qualibrate.core.config.resolvers import get_quam_state_path
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path

_config = get_qualibrate_config(get_qualibrate_config_path())
BACKEND = _config.project
_configured = get_quam_state_path(_config)
if not _configured:
    raise RuntimeError("No QUAM state path configured. Run install.py (workstation) or learn_setup.py (Learn) first.")
# install.py / learn_setup.py store an absolute path; a relative one (hand-run `qualibrate config`)
# means "relative to the checkout root" (the kernel's working directory).
STATE_PATH = Path(_configured)
if not STATE_PATH.is_absolute():
    STATE_PATH = Path.cwd() / STATE_PATH
print(f"backend: {BACKEND}\nstate:   {STATE_PATH}")


## 4 · Download the latest state and wiring
Uses `iqcc_cloud_client` (IQCC credentials must already be configured). **This overwrites the local `state.json` and `wiring.json`.**

In [ ]:
import json
from iqcc_cloud_client import IQCC_Cloud

qc = IQCC_Cloud(BACKEND)
latest_wiring = qc.state.get_latest("wiring")
latest_state = qc.state.get_latest("state")

STATE_PATH.mkdir(parents=True, exist_ok=True)
with open(STATE_PATH / "wiring.json", "w") as f:
    json.dump(latest_wiring.data, f, indent=4)
with open(STATE_PATH / "state.json", "w") as f:
    json.dump(latest_state.data, f, indent=4)
print("Wrote wiring.json and state.json to", STATE_PATH)

## 5 · Pick the qubit
A random one of the machine's active qubits, unless `QUBITS` is set in step 2.

In [ ]:
import json
import random

# Read from state.json, so this works wherever the checkout runs (no `config` import needed).
_state = json.loads((STATE_PATH / "state.json").read_text())
_active = _state.get("active_qubit_names") or list(_state.get("qubits", {}))
QUBITS_USED = QUBITS if QUBITS is not None else [random.choice(_active)]   # re-run this cell for another random qubit
print(f"node:   {NODE}\nqubits: {QUBITS_USED}   (active: {_active})")

## 6 · Run the node

<div style="background: rgba(56, 132, 255, 0.1); border-left: 4px solid rgba(56, 132, 255, 0.75); padding: 8px 14px; border-radius: 6px;">

💎 **QUA** — the node builds and runs a QUA program on the OPX (see `00_start_here.ipynb` §6 for how one is written).
🧩 **QUAlibrate** — `node.run(interactive=True)` runs the node: every action runs and the **data is saved** to `data/`, but the node's **state updates are only recorded as proposals** — the same propose-then-accept model as the QUAlibrate web app.
🧩 **QUAM** — `summary.state_updates` lists each proposed change as `path: old, new`.

</div>

In [ ]:
import pandas as pd
from IPython.display import display

node = library.nodes[NODE]
summary = node.run(interactive=True, qubits=QUBITS_USED, simulate=SIMULATE)

print("outcomes:", summary.outcomes, "| data saved as run id:", node.snapshot_idx)
for fig in node.results.get("figures", {}).values():
    display(fig)
pd.DataFrame(summary.state_updates).T   # proposed updates: old / new, not applied yet


## 7 · Accept the state updates (optional)
Nothing has been written to `state.json` yet. Uncomment the cell below to apply **all** proposed updates, or edit `ACCEPT` to list only some paths (copy them from the table above).
Note that a node often proposes related values together (for example `x180` and `x90` amplitudes), so accept them as a pair.

In [ ]:
# from qualibrate.core.utils.node.record_state_update import update_machine_attribute
#
# ACCEPT = list(summary.state_updates)   # all proposed updates; or only some paths copied from the table above
# machine = type(node.machine).load(str(STATE_PATH))   # fresh load of the current state
# for key in ACCEPT:
#     update_machine_attribute(machine, key, summary.state_updates[key]["new"])
# machine.save(str(STATE_PATH))
